In [0]:
%sql
create table teams (team string)

In [0]:
%sql
insert into teams(team) values ('aus'),('ind'),('wi'),('zim'),('sa')

In [0]:
%sql
select t1.team || ' vs ' || t2.team as matches from teams t1 join teams t2 on t1.team < t2.team

data = [ ('John', '2024-01-01', 100), ('John', '2024-01-05', 200), ('John', '2024-01-01', 400), ('John', '2024-01-05', 500), ('John', '2024-01-10', 300), ('Mary', '2024-01-01', 150), ('Mary', '2024-01-04', 250), ('Mary', '2024-01-09', 350) ] you need to compute the rolling sum of amounts for each customer over the last 3 transactions.
 
 
 
 
 schema ="Name string, Date string, amount int"


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
data = [ ('John', '2024-01-01', 100), ('John', '2024-01-05', 200), ('John', '2024-01-01', 400), ('John', '2024-01-05', 500), ('John', '2024-01-10', 300), ('Mary', '2024-01-01', 150), ('Mary', '2024-01-04', 250), ('Mary', '2024-01-09', 350) ] 
schema ="Name string, Date string, amount int"

In [0]:
data_df = spark.createDataFrame(data,schema)
data_df.display()

In [0]:
windowspec = Window.partitionBy("Name").orderBy(desc("date"))
cust_last_3_tran_df =  data_df.withColumn("row_number", row_number().over(windowspec)).filter("row_number < 4")
cust_last_3_tran_df.display()

In [0]:
windowspec = Window.partitionBy("Name").orderBy(desc("date")).rowsBetween(Window.unboundedPreceding, Window.currentRow)
res_df = cust_last_3_tran_df.withColumn("rolling_sum", sum("amount").over(windowspec))
res_df.display()

In [0]:
%sql
-- Create the Employee table
CREATE TABLE  if not exists Employee1(
    emp_id INT PRIMARY KEY,
    name VARCHAR(50),
    salary INT,
    manager_id INT
);

-- Insert sample data
INSERT INTO Employee1 (emp_id, name, salary, manager_id) VALUES
(1, 'Alice', 5000, 3),
(2, 'Bob', 6000, 3),
(3, 'Carol', 7000, NULL),
(4, 'David', 4500, 2);


In [0]:
%sql
select * from employee1

In [0]:
%sql
select e1.name as employee_name, m1.name as manager_name from employee1 e1 join employee1 m1 on e1.manager_id = m1.emp_id

In [0]:
df = spark.sql("select * from sales_data")

In [0]:
df.display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
agg_df = df.groupBy("region","product").agg(sum("revenue").alias("Total_revenue"))
agg_df.display()

In [0]:
Windowspec = Window.partitionBy("region").orderBy(desc("Total_revenue"))
res_df = agg_df.withColumn("Ranks", dense_rank().over(Windowspec))
res_df.display()

In [0]:
Windowspec = Window.partitionBy("region").orderBy(desc("Total_revenue"))
res_df = agg_df.withColumn("row_number", row_number().over(Windowspec)).filter("row_number < 2")
res_df.display()

In [0]:
df.display()

In [0]:
windowspec = Window.partitionBy("region").orderBy("year").rowsBetween(Window.unboundedPreceding, Window.currentRow)
res_df = df.withColumn("running_total", sum("revenue").over(windowspec))
res_df.display()

**YOY growth**

In [0]:
df.display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
year_df = df.groupBy("region", "year").agg(sum("revenue").alias("Total_revenue_by_year"))
year_df.display()

In [0]:
Windowspec = Window.partitionBy("region").orderBy("year")
prev_year_df = year_df.withColumn("prev_year", lag("Total_revenue_by_year").over(Windowspec))
prev_year_df.display()

In [0]:
yoy_df = prev_year_df.withColumn("YOY_Growth", round(((col("Total_revenue_by_year") - col("prev_year")) / col("prev_year") * 100),2))
yoy_df.display()

**Moving average**

In [0]:
year_df.display()

In [0]:
Windowspec = Window.partitionBy("region").orderBy("year").rowsBetween(-2,0)
res_df = year_df.withColumn("moving_avg", avg("Total_revenue_by_year").over(Windowspec))
res_df.display()